# Confidence calibration on MMLU-Pro with normalized-entropy uncertainty

**Objective.** Measure how closely each model's stated confidence matches its empirical
accuracy on a standard multiple-choice benchmark (MMLU-Pro), and how that relationship
varies across *subject category* and *question difficulty*.

This is the *standard* counterpart to `sim_uncertainty_calibration_target.ipynb`. The pipeline
is identical (same models, same caching, same calibration metrics and two-level→bootstrap
machinery) **except for how the uncertainty score is computed**. MMLU-Pro items do not all
have the same number of options (they range from 3 to 10), so the raw KnowNo confidence (the
probability mass on the arg-max option, Ren et al. 2023) is not comparable across questions:
a max-probability of 0.3 means something very different over 3 options than over 10. We
therefore score uncertainty with the **normalized Shannon entropy** of the option
distribution, the entropy divided by `log(#options)`, which lands in `[0, 1]` regardless of
how many options a question has (0 = all mass on one option, 1 = uniform). The calibration
confidence used everywhere below is the derived **certainty** `1 - normalized_entropy`.

Each option's probability is the model's next-token probability for its letter, normalised
over the question's letters (A…). Difficulty is labelled by an independent **five-LLM panel**
(none of them a model under test) that answers each question once with no reasoning; a
question's *panel solvability* (0-5 correct) is the difficulty proxy. Calibration is summarised
with accuracy, ECE, reliability curves and mean confidence, with bootstrap confidence intervals.

## 1. Configuration

In [ ]:
# Pin BLAS/OpenMP threads to 1 before importing the pipeline (numpy/open3d stack otherwise
# spins on this machine). Must precede the imports below.
import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = "1"

import re, json, math, random, hashlib, time
from dataclasses import dataclass
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy

from datasets import load_dataset
from reflect.llm.prompter import LocalLLMPrompter, PortkeyLLMPrompter
from reflect.core.paths import analysis_experiment_dir

DATASET = "TIGER-Lab/MMLU-Pro"
SPLIT   = "all"
OUT_DIR = analysis_experiment_dir("llm_calibration_standard_entropy")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Models under test; calibration is computed and compared for each.
MUT_MODELS = ["qwen3.5:9b", "qwen3.5:27b", "gpt-5.4", "qwen3.6:27b", "qwen3.6:35b"]
# Five-LLM difficulty panel (none of them a model under test), answering with no reasoning.
PANEL_MODELS = ["mistral:7b", "llama3.2:3b", "granite3.3:8b", "phi4-mini:3.8b", "command-r7b"]

MODEL_BACKENDS = {
    "qwen3.5:9b":  {"backend": "local"},
    "qwen3.5:27b": {"backend": "local"},
    "gpt-5.4":     {"backend": "portkey", "reasoning_effort": "none", "virtual_key_env": "PORTKEY_VIRTUAL_KEY"},
    "qwen3.6:27b": {"backend": "local"},
    "qwen3.6:35b": {"backend": "local"},
}
OLLAMA_URL = "http://localhost:11434"

LETTERS       = "ABCDEFGHIJ"   # MMLU-Pro has up to 10 options
SEED          = 16
ECE_BINS      = 10             # equal-width confidence bins for ECE / reliability
N_BOOT        = 2000           # bootstrap resamples
MAX_QUESTIONS = None           # int -> stratified subsample by category; None -> all questions

# Difficulty bucket from panel solvability (number of correct panel answers, 0..5).
def difficulty_bucket(solvability):
    if solvability <= 1: return "hard"
    if solvability <= 3: return "medium"
    return "easy"

print("models under test:", MUT_MODELS)
print("difficulty panel :", PANEL_MODELS)
print("output dir       :", OUT_DIR)

## 2. Dataset

MMLU-Pro is a 12k-item multiple-choice benchmark across 14 subject categories. Unlike plain
MMLU, the number of options varies per question (3–10), which is exactly why uncertainty has
to be normalised by the option count to be comparable across items.

In [ ]:
ds = load_dataset(DATASET, split=SPLIT).to_pandas()
print(f"loaded {len(ds)} questions across {ds['category'].nunique()} categories")
print("option-count distribution:", dict(sorted(Counter(ds["options"].apply(len)).items())))
ds.info()

In [ ]:
@dataclass
class Question:
    qid: int; category: str; stem: str
    options: dict; answer: str; letters: list; n_options: int

def build_questions(df):
    qs = []
    for r in df.itertuples(index=False):
        opts = list(r.options)
        letters = list(LETTERS[:len(opts)])
        qs.append(Question(int(r.question_id), r.category, r.question,
                           {L: str(opts[i]) for i, L in enumerate(letters)},
                           r.answer, letters, len(opts)))
    return qs

questions = build_questions(ds)

# Optional stratified-by-category subsample. The on-disk cache makes a full run resumable, so
# leave MAX_QUESTIONS = None to score every question; set an int for a quick smoke test.
if MAX_QUESTIONS is not None and MAX_QUESTIONS < len(questions):
    by_cat = defaultdict(list)
    for q in questions: by_cat[q.category].append(q)
    cats = sorted(by_cat); per = max(1, MAX_QUESTIONS // len(cats))
    rng = random.Random(SEED); picked = []
    for c in cats:
        pool = sorted(by_cat[c], key=lambda q: q.qid); rng.shuffle(pool)
        picked.extend(pool[:per])
    rng.shuffle(picked)
    questions = picked[:MAX_QUESTIONS]

assert len({q.qid for q in questions}) == len(questions), "question ids must be unique"
print(f"using {len(questions)} questions")
print("by category:", dict(sorted(Counter(q.category for q in questions).items())))

## 3. Model interface and the normalized-entropy uncertainty score

Each option's score is the model's next-token probability for its letter, normalised over the
question's letters (KnowNo). From that option distribution we record two quantities:

- **`max_prob`** = the probability mass on the arg-max option (the raw KnowNo confidence, kept
  for reference / comparison with the *target* notebook).
- **`norm_entropy`** = the Shannon entropy of the option distribution divided by `log(#options)`,
  so it is in `[0, 1]` and comparable across questions with different option counts. The
  **calibration confidence** is the derived certainty `confidence = 1 - norm_entropy`.

Sampling is greedy (`temperature=0`, one token), so calls are deterministic, and responses are
cached on disk keyed by `(model, prompt)` so the experiment is cheap to re-run.

In [ ]:
ANSWER_SYS = ("You answer multiple-choice questions. "
              "Read the question and output ONLY the single letter of the best option. "
              "No words, no punctuation, no explanation.")

def build_user(q):
    opts = "\n".join(f"{L} = {q.options[L]}" for L in q.letters)
    return f"Question: {q.stem}\n{opts}\nAnswer:"

def normalized_entropy(option_probs):
    '''Shannon entropy of the option distribution, normalised to [0, 1] by log(#options).

    Dividing by log(#options) makes uncertainty comparable across questions with different
    numbers of options (MMLU-Pro items have 3-10): 0 = all probability mass on one option,
    1 = uniform over the options. Returns None if no distribution is available.'''
    if not option_probs:
        return None
    p = np.asarray(list(option_probs.values()), dtype=float)
    n = p.size
    if n <= 1:
        return 0.0
    p = p[p > 0]
    if p.size == 0:
        return None
    raw = float(scipy_entropy(p, base=np.e))   # nats
    return raw / math.log(n)

def build_prompter(model):
    '''Return a prompter for a model. qwen (local) and gpt-5.4 (portkey) score option
    probabilities from logprobs. Raises if a required API key is missing.'''
    cfg = MODEL_BACKENDS.get(model, {"backend": "local"})
    backend = cfg["backend"]
    if backend == "local":
        return LocalLLMPrompter(model_name=model, base_url=OLLAMA_URL, think=False)
    if backend == "portkey":
        key = os.environ.get("PORTKEY_API_KEY")     # set in your shell or .env (see .env.example)
        if not key: raise RuntimeError("PORTKEY_API_KEY not set")
        vk = os.environ.get(cfg.get("virtual_key_env", "")) or cfg.get("virtual_key")
        return PortkeyLLMPrompter(portkey_api_key=key, model=model, virtual_key=vk,
                                  reasoning_effort=cfg.get("reasoning_effort"))
    raise ValueError(f"unknown backend {backend!r}")

_CACHE_PATH = OUT_DIR / "llm_cache.json"
_LLM_CACHE = json.loads(_CACHE_PATH.read_text()) if _CACHE_PATH.exists() else {}

def _cache_key(model, user):
    return hashlib.sha256(f"{model}\x00{ANSWER_SYS}\x00{user}".encode()).hexdigest()

def _flush_cache():
    _CACHE_PATH.write_text(json.dumps(_LLM_CACHE))

def ask(prompter, model, q, user=None):
    '''Score one question. Returns predicted_label, option_probs, max_prob (KnowNo confidence),
    norm_entropy, certainty (= 1 - norm_entropy, the calibration confidence), score_status,
    correct. The raw logprob-derived fields are cached; norm_entropy/certainty are derived on
    read so the entropy definition can be changed without re-querying the models.'''
    user = build_user(q) if user is None else user
    key = _cache_key(model, user)
    if key not in _LLM_CACHE:
        cs = {L: q.options[L] for L in q.letters}
        try:
            _txt, meta = prompter.query(prompt={"system": ANSWER_SYS, "user": user},
                                        sampling_params={"max_tokens": 1, "temperature": 0.0, "top_p": 1},
                                        choice_spec=cs)
            _LLM_CACHE[key] = {"predicted_label": meta.get("predicted_label"),
                               "max_prob": meta.get("confidence"),
                               "option_probs": meta.get("option_probs"),
                               "score_status": meta.get("score_status")}
        except Exception as e:
            _LLM_CACHE[key] = {"predicted_label": None, "max_prob": None,
                               "option_probs": None, "score_status": f"error:{type(e).__name__}"}
    m = _LLM_CACHE[key]
    ne = normalized_entropy(m["option_probs"]) if m["option_probs"] else None
    return {"predicted_label": m["predicted_label"], "max_prob": m["max_prob"],
            "option_probs": m["option_probs"], "score_status": m["score_status"],
            "norm_entropy": ne, "certainty": (None if ne is None else 1.0 - ne),
            "correct": int(m["predicted_label"] == q.answer) if m["predicted_label"] else 0}

def run_model_over_questions(model, qs, label):
    '''Iterate one model over all questions, reusing (and periodically flushing) the cache.'''
    prompter = build_prompter(model)
    t0 = time.time(); res = {}
    for k, q in enumerate(qs):
        res[q.qid] = ask(prompter, model, q)
        if (k + 1) % 100 == 0:
            _flush_cache(); print(f"  [{label}] {k+1}/{len(qs)}", flush=True)
    _flush_cache()
    print(f"  [{label}] done {len(qs)} in {time.time()-t0:.0f}s")
    return res

## 4. Difficulty labelling (five-LLM panel)

A panel of five LLMs (none of them a model under test) answers each question once with no
reasoning steps. A question's **panel solvability** is the number of correct panel answers
(0-5), used as a difficulty proxy for contemporary LLMs and bucketed into hard / medium / easy.

In [ ]:
panel_correct = defaultdict(int)
for model in PANEL_MODELS:
    print(f"panel: {model}")
    res = run_model_over_questions(model, questions, model)
    i = 0
    for qid, r in res.items():
        panel_correct[qid] += r["correct"]
        i += 1
        if i % 100 == 0:
            print(f"  [{model}] {i}/{len(questions)}", flush=True)
        

solvability = {q.qid: panel_correct[q.qid] for q in questions}
print("\nsolvability distribution (0..5):",
      dict(sorted(Counter(solvability.values()).items())))
print("difficulty buckets:",
      dict(sorted(Counter(difficulty_bucket(s) for s in solvability.values()).items())))


In [ ]:
# Store panel solvability in a JSON file
SOLVABILITY_PATH = OUT_DIR / "panel_solvability_standard.json"
with SOLVABILITY_PATH.open("w") as f:
    json.dump(solvability, f, indent=2)
print(f"panel solvability written to {SOLVABILITY_PATH}")

## 5. Stratified Sub-sampling

In [ ]:
TARGET_N, MIN_PER_CELL, STRAT_SEED = 1500, 15, 16
SAMPLE_PATH = OUT_DIR / "mut_sample.json"
full_questions = list(questions)                      # keep the full set

def _draw_stratified(qs, solv, target_n, floor, seed):
    diff = {q.qid: difficulty_bucket(solv[q.qid]) for q in qs}
    strata = defaultdict(list)
    for q in qs: strata[(q.category, diff[q.qid])].append(q)
    N, rng, picked, alloc = len(qs), random.Random(seed), [], {}
    for k, v in strata.items():
        alloc[k] = min(len(v), max(floor, round(target_n * len(v) / N)))
    for k, v in strata.items():
        pool = sorted(v, key=lambda q: q.qid); rng.shuffle(pool); picked.extend(pool[:alloc[k]])
    rng.shuffle(picked)
    nsel = Counter((q.category, diff[q.qid]) for q in picked)   # inverse-prob (population) weights
    w = {q.qid: len(strata[(q.category, diff[q.qid])]) / nsel[(q.category, diff[q.qid])] for q in picked}
    return picked, w

if SAMPLE_PATH.exists():
    blob = json.loads(SAMPLE_PATH.read_text()); keep = set(blob["qids"])
    sample_weight = {int(k): v for k, v in blob["weights"].items()}
    questions = [q for q in full_questions if q.qid in keep]
    print(f"loaded existing sample: {len(questions)} questions")
else:
    questions, sample_weight = _draw_stratified(full_questions, solvability, TARGET_N, MIN_PER_CELL, STRAT_SEED)
    SAMPLE_PATH.write_text(json.dumps({"qids": [q.qid for q in questions],
                                       "weights": {str(k): v for k, v in sample_weight.items()}}))
    print(f"drew stratified sample: {len(questions)} questions, saved to {SAMPLE_PATH.name}")

_s = pd.DataFrame([(q.category, difficulty_bucket(solvability[q.qid])) for q in questions],
                  columns=["category", "difficulty"])
print(pd.crosstab(_s.category, _s.difficulty, margins=True).to_string())


## 6. Models under test

Each model under test answers every question. For each answer we record the option
probabilities, the arg-max `max_prob`, the `norm_entropy`, the derived calibration
`confidence` (= 1 - norm_entropy), and whether the chosen option is correct.

In [ ]:
all_rows = []
for model in MUT_MODELS:
    try:
        res = run_model_over_questions(model, questions, model)
    except Exception as e:
        print(f"  [skip] {model}: {type(e).__name__}: {e}")
        continue
    for q in questions:
        r = res[q.qid]; solv = solvability[q.qid]
        all_rows.append({"model": model, "qid": q.qid, "category": q.category,
                         "n_options": q.n_options, "answer": q.answer,
                         "predicted": r["predicted_label"], "max_prob": r["max_prob"],
                         "norm_entropy": r["norm_entropy"], "confidence": r["certainty"],
                         "correct": r["correct"], "score_status": r["score_status"],
                         "solvability": solv, "difficulty": difficulty_bucket(solv)})

df_all = pd.DataFrame(all_rows)
df_all.to_csv(OUT_DIR / "results_all_models.csv", index=False)
print("per-model coverage:")
print(df_all.groupby("model").agg(
        n=("qid", "size"),
        with_conf=("confidence", lambda s: s.notna().sum()),
        accuracy=("correct", "mean"),
        mean_entropy=("norm_entropy", "mean")).round(3).to_string())

df_cal = df_all[df_all["confidence"].notna()].copy()
df_cal.head()

In [ ]:
df_all["weight"] = df_all["qid"].map(sample_weight)
_miss = int(df_all["weight"].isna().sum())
if _miss:
    print(f"WARNING: {_miss} rows had no sample weight (qid outside the stratified "
          "sample -- stale results_all_models.csv?); filling with median weight. "
          "Re-run section 5 so `questions` == the sample for a clean weighting.")
    df_all["weight"] = df_all["weight"].fillna(df_all["weight"].median())
df_cal = df_all[df_all["confidence"].notna()].copy()
_w = df_cal["weight"].values
_e = (_w.sum() ** 2) / np.sum(_w ** 2)
print(f"weights attached to {len(df_cal)} scored rows; Kish ESS={_e:.0f} "
      f"(ESS/n={_e/len(df_cal):.3f}, design effect={len(df_cal)/_e:.2f}); "
      "~1 => weighting barely changes variance here.")


In [ ]:
# Retry ONLY the gpt-5.4 questions that failed (cached errors / missing / no probability).
MODEL = "gpt-5.4"
RETRIES = 5                  # attempts per question (transient APIConnectionError, etc.)

def _needs_redo(entry):
    return (entry is None
            or str(entry.get("score_status", "")).startswith("error")
            or entry.get("max_prob") is None)

todo = [q for q in questions if _needs_redo(_LLM_CACHE.get(_cache_key(MODEL, build_user(q))))]
print(f"{len(todo)} gpt-5.4 questions need redo (of {len(questions)})")

if todo:
    prompter = build_prompter(MODEL)
    sp = {"max_tokens": 1, "temperature": 0.0, "top_p": 1}
    fixed, still_failing = 0, []
    for i, q in enumerate(todo):
        key = _cache_key(MODEL, build_user(q))
        cs = {L: q.options[L] for L in q.letters}
        err = None
        for attempt in range(RETRIES):
            try:
                _txt, meta = prompter.query(prompt={"system": ANSWER_SYS, "user": build_user(q)},
                                            sampling_params=sp, choice_spec=cs)
                _LLM_CACHE[key] = {"predicted_label": meta.get("predicted_label"),
                                   "max_prob":        meta.get("confidence"),
                                   "option_probs":    meta.get("option_probs"),
                                   "score_status":    meta.get("score_status")}
                err = None; fixed += 1; break
            except Exception as e:
                err = e; time.sleep(2 * (attempt + 1))     # 2s, 4s, 6s, 8s, 10s backoff
        if err is not None:
            still_failing.append((q.qid, type(err).__name__))
        if (i + 1) % 25 == 0:
            _flush_cache(); print(f"  {i+1}/{len(todo)} done")
    _flush_cache()
    print(f"redo complete: {fixed}/{len(todo)} now answered; {len(still_failing)} still failing")
    if still_failing:
        print("still failing:", still_failing[:10])

## 7. Calibration metrics and bootstrap

Expected Calibration Error (ECE) bins predictions by confidence and averages the
|accuracy - confidence| gap weighted by bin count; the reliability curve plots accuracy against
mean confidence per bin. Confidence intervals come from a nonparametric bootstrap over
questions.

In [ ]:
def ece(conf, correct, n_bins=ECE_BINS):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    if len(conf) == 0: return np.nan
    edges = np.linspace(0, 1, n_bins + 1); total = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        m = (conf > lo) & (conf <= hi) if b > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0: continue
        total += (m.sum() / len(conf)) * abs(correct[m].mean() - conf[m].mean())
    return total

def reliability_curve(conf, correct, n_bins=ECE_BINS):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    edges = np.linspace(0, 1, n_bins + 1); pts = []
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        m = (conf > lo) & (conf <= hi) if b > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0: continue
        pts.append((conf[m].mean(), correct[m].mean(), int(m.sum())))
    return pts  # list of (mean_conf, accuracy, count)

def m_accuracy(d): return d["correct"].mean() if len(d) else np.nan
def m_meanconf(d): return d["confidence"].mean() if len(d) else np.nan
def m_ece(d):      return ece(d["confidence"].values, d["correct"].values) if len(d) else np.nan

def bootstrap_ci(d, metric_fn, B=N_BOOT, seed=SEED):
    '''Nonparametric bootstrap over questions; returns (2.5, 50, 97.5) percentiles.'''
    if len(d) == 0: return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed); arr = d.reset_index(drop=True); n = len(arr); vals = []
    for _ in range(B):
        bs = arr.iloc[rng.integers(0, n, n)]
        v = metric_fn(bs)
        if v is not None and not (isinstance(v, float) and math.isnan(v)):
            vals.append(v)
    if not vals: return (np.nan, np.nan, np.nan)
    return tuple(np.percentile(vals, [2.5, 50, 97.5]))

def summarise(d):
    out = {"n": len(d), "accuracy": m_accuracy(d), "mean_conf": m_meanconf(d), "ece": m_ece(d),
           "mean_entropy": d["norm_entropy"].mean() if len(d) else np.nan,
           "gap": (m_meanconf(d) - m_accuracy(d))}
    for name, fn in [("accuracy", m_accuracy), ("ece", m_ece)]:
        lo, mid, hi = bootstrap_ci(d, fn)
        out[f"{name}_lo"], out[f"{name}_hi"] = lo, hi
    return out

print(f"overall (pooled): accuracy={m_accuracy(df_cal):.3f}  "
      f"mean_conf={m_meanconf(df_cal):.3f}  ECE={m_ece(df_cal):.3f}  "
      f"mean_entropy={df_cal['norm_entropy'].mean():.3f}")

In [ ]:
EPS = 1e-12

def wmean(x, w=None):
    x = np.asarray(x, float)
    if w is None: return float(x.mean()) if x.size else np.nan
    w = np.asarray(w, float)
    return np.nan if w.sum() == 0 else float(np.average(x, weights=w))

def ess(w):
    """Kish effective sample size; ESS/n = 1/design-effect quantifies variance inflation."""
    w = np.asarray(w, float)
    return float(w.sum() ** 2 / np.sum(w ** 2)) if w.sum() > 0 else 0.0

def ece(conf, correct, n_bins=ECE_BINS, weight=None):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    if len(conf) == 0: return np.nan
    w = np.ones_like(conf) if weight is None else np.asarray(weight, float)
    W = w.sum()
    if W == 0: return np.nan
    edges = np.linspace(0, 1, n_bins + 1); total = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        m = (conf > lo) & (conf <= hi) if b > 0 else (conf >= lo) & (conf <= hi)
        wb = w[m].sum()
        if wb == 0: continue
        total += (wb / W) * abs(wmean(correct[m], w[m]) - wmean(conf[m], w[m]))
    return total

def brier(conf, correct, weight=None):
    conf = np.asarray(conf, float); y = np.asarray(correct, float)
    if len(conf) == 0: return np.nan
    return wmean((conf - y) ** 2, None if weight is None else np.asarray(weight, float))

def nll(conf, correct, weight=None):
    conf = np.clip(np.asarray(conf, float), EPS, 1 - EPS); y = np.asarray(correct, float)
    if len(conf) == 0: return np.nan
    ll = y * np.log(conf) + (1 - y) * np.log(1 - conf)
    return -wmean(ll, None if weight is None else np.asarray(weight, float))

# dataframe wrappers (the slice must carry a 'weight' column); w=True -> weighted
def m_accuracy(d, w=False): return wmean(d["correct"].values, d["weight"].values if w else None)
def m_meanconf(d, w=False): return wmean(d["confidence"].values, d["weight"].values if w else None)
def m_ece(d, w=False):      return ece(d["confidence"].values, d["correct"].values, weight=d["weight"].values if w else None)
def m_brier(d, w=False):    return brier(d["confidence"].values, d["correct"].values, weight=d["weight"].values if w else None)
def m_nll(d, w=False):      return nll(d["confidence"].values, d["correct"].values, weight=d["weight"].values if w else None)

_BOOT = bootstrap_ci   # bootstrap function defined in the metrics cell above

def summarise(d):
    out = {"n": len(d), "ess": round(ess(d["weight"].values), 1),
           "accuracy": m_accuracy(d), "mean_conf": m_meanconf(d),
           "ece": m_ece(d), "brier": m_brier(d), "nll": m_nll(d),
           "ece_w": m_ece(d, True), "brier_w": m_brier(d, True), "nll_w": m_nll(d, True),
           "mean_entropy": d["norm_entropy"].mean() if (len(d) and "norm_entropy" in d) else np.nan,
           "gap": (m_meanconf(d) - m_accuracy(d))}
    for name, fn in [("accuracy", m_accuracy), ("ece", m_ece)]:
        lo, mid, hi = _BOOT(d, fn)
        out[f"{name}_lo"], out[f"{name}_hi"] = lo, hi
    return out

print("proper-scoring metrics ready: ece (now weight-capable), brier, nll; "
      "summarise() now reports brier/nll and weighted variants.")


## 8. Results by category and by difficulty bucket

In [ ]:
def marginal_table(group_col, order=None):
    groups = order if order is not None else sorted(df_cal[group_col].unique())
    rws = []
    for g in groups:
        d = df_cal[df_cal[group_col] == g]
        if len(d) == 0: continue
        rws.append({group_col: g, **summarise(d)})
    return pd.DataFrame(rws)

by_cat  = marginal_table("category")
by_diff = marginal_table("difficulty", ["hard", "medium", "easy"])

def show(tbl, title):
    print(title)
    cols = [tbl.columns[0], "n", "accuracy", "mean_conf", "ece", "mean_entropy", "gap"]
    disp = tbl[cols].copy()
    for c in ["accuracy", "mean_conf", "ece", "mean_entropy", "gap"]:
        disp[c] = disp[c].round(3)
    print(disp.to_string(index=False)); print()

show(by_cat,  "By category (pooled across models; gap = mean confidence - accuracy):")
show(by_diff, "By difficulty bucket (panel solvability):")

## 9. Reliability diagrams and calibration figures

In [ ]:
# Multi-model setup for all figures below (only answers with a usable confidence enter here).
cal_all = df_all[df_all["confidence"].notna()].copy()
present = [m for m in MUT_MODELS if m in cal_all["model"].unique()]
MCOLOR  = {m: c for m, c in zip(MUT_MODELS, ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"])}
DIFFS   = ["hard", "medium", "easy"]
DIFF_NAME = {"hard": "(solv<=1)", "medium": "(solv 2-3)", "easy": "(solv>=4)"}
print("models present in figures:", present)

In [ ]:
# FIG: reliability grid (model rows x difficulty cols). Marker AREA is scaled by the number of
#    items in each confidence bin, so sparse bins (little evidence) look small. ECE already
#    weights bins by count, so the plot and the summary metric agree.
def reliability_grid(facet, levels, level_name, fname, title):
    nrow, ncol = len(present), len(levels)
    smax = 1
    for m in present:
        for lv in levels:
            d = cal_all[(cal_all.model == m) & (cal_all[facet] == lv)]
            for _, _, c in reliability_curve(d.confidence.values, d.correct.values): smax = max(smax, c)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.1*ncol, 3.1*nrow), squeeze=False)
    for r, m in enumerate(present):
        for c, lv in enumerate(levels):
            ax = axes[r][c]; d = cal_all[(cal_all.model == m) & (cal_all[facet] == lv)]
            ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
            pts = reliability_curve(d.confidence.values, d.correct.values)
            if pts:
                xs, ys, cs = zip(*pts)
                ax.plot(xs, ys, "-", color=MCOLOR[m], lw=1, alpha=.5, zorder=2)
                ax.scatter(xs, ys, s=[25 + 275*(cc/smax) for cc in cs], color=MCOLOR[m],
                           edgecolor="white", linewidth=.5, zorder=3)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
            ax.set_title(f"acc={d.correct.mean():.2f} conf={d.confidence.mean():.2f}\n"
                         f"ECE={ece(d.confidence.values, d.correct.values):.2f} n={len(d)}", fontsize=8)
            if c == 0: ax.set_ylabel(f"{m}\naccuracy", fontsize=9)
            if r == nrow-1: ax.set_xlabel(f"confidence (1-norm.entropy)\n{lv} {level_name.get(lv,'')}", fontsize=9)
            ax.tick_params(labelsize=7)
    fig.suptitle(title + "  (marker area ∝ #items in bin)", y=1.002, fontsize=12)
    fig.tight_layout(); fig.savefig(OUT_DIR / fname, dpi=130, bbox_inches="tight"); plt.show()

reliability_grid("difficulty", DIFFS, DIFF_NAME, "reliability_grid_difficulty.png",
                 "Reliability by model x difficulty")

In [ ]:
# FIG: accuracy / mean confidence / mean normalized-entropy / ECE vs difficulty, all models.
solv = sorted(cal_all.solvability.unique())
fig, axes = plt.subplots(1, 4, figsize=(20, 4.4))
panels = [("accuracy",         lambda d: d.correct.mean(),                       True),
          ("mean confidence",  lambda d: d.confidence.mean(),                    True),
          ("mean norm.entropy",lambda d: d.norm_entropy.mean(),                  True),
          ("ECE",              lambda d: ece(d.confidence.values, d.correct.values), False)]
for ax, (metric, fn, clip01) in zip(axes, panels):
    for m in present:
        ys = [fn(cal_all[(cal_all.model == m) & (cal_all.solvability == s)]) for s in solv]
        ax.plot(solv, ys, "-o", color=MCOLOR[m], label=m)
    ax.set_xlabel("panel solvability (0 = hardest, 5 = easiest)"); ax.set_title(metric); ax.grid(alpha=.3)
    ax.set_ylim(0, 1.02) if clip01 else ax.set_ylim(0, None)
ns = [int((cal_all[cal_all.model == present[0]].solvability == s).sum()) for s in solv] if present else []
axes[0].legend(fontsize=8)
if present:
    axes[0].set_xticks(solv); axes[0].set_xticklabels([f"{s}\n(n={n})" for s, n in zip(solv, ns)], fontsize=8)
fig.suptitle("Accuracy / confidence / normalized-entropy / ECE vs difficulty, by model", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(OUT_DIR / "metrics_vs_difficulty_by_model.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# FIG: ECE heatmap (model x category) + calibration map scatter (one point per model x category).
cats = sorted(cal_all.category.unique())

H = np.full((len(present), len(cats)), np.nan)
for ri, m in enumerate(present):
    for ci, cl in enumerate(cats):
        d = cal_all[(cal_all.model == m) & (cal_all.category == cl)]
        if len(d): H[ri, ci] = ece(d.confidence.values, d.correct.values)
fig, ax = plt.subplots(figsize=(0.9*len(cats)+2, 0.7*len(present)+2))
im = ax.imshow(H, cmap="OrRd", vmin=0, vmax=np.nanmax(H) if np.isfinite(H).any() else 1)
ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(present))); ax.set_yticklabels(present, fontsize=9)
for i in range(len(present)):
    for j in range(len(cats)):
        if np.isfinite(H[i, j]):
            ax.text(j, i, f"{H[i,j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if H[i, j] > 0.5*np.nanmax(H) else "black")
fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label="ECE")
ax.set_title("ECE by model x category (darker = more miscalibrated)", fontsize=11)
fig.tight_layout(); fig.savefig(OUT_DIR / "ece_heatmap_by_category.png", dpi=130, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(6.2, 6))
ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="perfect calibration")
for m in present:
    xs, ys, ss = [], [], []
    for cl in cats:
        d = cal_all[(cal_all.model == m) & (cal_all.category == cl)]
        if len(d): xs.append(d.correct.mean()); ys.append(d.confidence.mean()); ss.append(len(d))
    ax.scatter(xs, ys, s=[20 + 0.05*n for n in ss], color=MCOLOR[m], alpha=.7,
               edgecolor="white", linewidth=.5, label=m)
ax.set_xlabel("accuracy (per category)"); ax.set_ylabel("mean confidence (per category)")
ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02); ax.set_aspect("equal"); ax.legend(fontsize=8); ax.grid(alpha=.3)
ax.set_title("Calibration map: each point = one (model, category)\nabove diagonal = overconfident; size ∝ #items")
fig.tight_layout(); fig.savefig(OUT_DIR / "calibration_scatter.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# FIG: distribution of the normalized-entropy uncertainty per model, split by correctness.
#    A well-behaved uncertainty score puts correct answers at low entropy and errors at high.
fig, axes = plt.subplots(1, len(present), figsize=(5*max(len(present),1), 3.6), sharey=True, squeeze=False)
bins = np.linspace(0, 1, 21)
for ax, m in zip(axes[0], present):
    d = cal_all[cal_all.model == m]
    ax.hist(d[d.correct == 1].norm_entropy, bins=bins, alpha=.7, color="#2ca02c", label="correct")
    ax.hist(d[d.correct == 0].norm_entropy, bins=bins, alpha=.7, color="#d62728", label="wrong")
    ax.set_title(f"{m}\nmean entropy={d.norm_entropy.mean():.2f}", fontsize=10)
    ax.set_xlabel("normalized entropy (uncertainty)"); ax.set_xlim(0, 1); ax.legend(fontsize=8)
axes[0][0].set_ylabel("# questions")
fig.suptitle("Normalized-entropy uncertainty by correctness", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(OUT_DIR / "entropy_by_correctness.png", dpi=130, bbox_inches="tight"); plt.show()

## 10. Cross-model comparison

Calibration of every model under test on the shared question set, using the normalized-entropy
confidence. Only answers with a usable confidence value enter the calibration columns.

In [ ]:
cmp_rows = []
for m in present:
    d = cal_all[cal_all.model == m]
    cmp_rows.append({"model": m, "n": len(d), "accuracy": d.correct.mean(),
                     "mean_conf": d.confidence.mean(), "mean_entropy": d.norm_entropy.mean(),
                     "ece": m_ece(d), "gap": d.confidence.mean() - d.correct.mean()})
cross = pd.DataFrame(cmp_rows)
cross.to_csv(OUT_DIR / "metrics_by_model.csv", index=False)
print("Overall calibration by model (gap = mean confidence - accuracy):")
print(cross.round(3).to_string(index=False))

print("\nECE by model x category:")
ece_mc = pd.DataFrame(index=present, columns=sorted(cal_all.category.unique()), dtype=float)
for m in present:
    for cl in ece_mc.columns:
        d = cal_all[(cal_all.model == m) & (cal_all.category == cl)]
        ece_mc.loc[m, cl] = m_ece(d) if len(d) else np.nan
print(ece_mc.round(3).to_string())

### Proper scoring rules vs ECE, weighted and unweighted

Per-model calibration under ECE and two strictly proper scores (Brier, NLL), each
computed unweighted and under the distribution weights (`*_w`). If Brier and NLL
induce the **same model ordering** as ECE, the ECE-based conclusion is not a binning
artefact. `ess` is the Kish effective sample size after weighting (n/ess = design
effect = variance inflation).

In [ ]:
# === ECE vs proper scores: per-model ranking, weighted & unweighted ==========
def _rank(s):                      # 1 = best (lowest error)
    return s.rank(method="min").astype(int)

_rows = []
for m in present:
    d = cal_all[cal_all.model == m]
    _rows.append({"model": m, "n": len(d), "ess": round(ess(d["weight"].values), 1),
                  "accuracy": m_accuracy(d), "mean_conf": m_meanconf(d),
                  "ece": m_ece(d), "brier": m_brier(d), "nll": m_nll(d),
                  "ece_w": m_ece(d, True), "brier_w": m_brier(d, True), "nll_w": m_nll(d, True)})
score = pd.DataFrame(_rows).set_index("model")
print("Per-model calibration (lower ece/brier/nll = better calibrated):")
print(score.round(4).to_string())

_metrics = ["ece", "brier", "nll", "ece_w", "brier_w", "nll_w"]
ranks = score[_metrics].apply(_rank)
print("\nModel ranking by each metric (1 = best calibrated):")
print(ranks.to_string())

ref = "ece"
print(f"\nDoes each metric induce the SAME ordering as unweighted ECE?")
for c in _metrics:
    if c == ref: continue
    same = bool((ranks[c] == ranks[ref]).all())
    print(f"  {c:8s}: {'AGREES ' if same else 'DIFFERS'}  order={list(score[c].sort_values().index)}")

score.to_csv(OUT_DIR / "metrics_by_model_scored.csv")
print("\nsaved", OUT_DIR / "metrics_by_model_scored.csv")


In [ ]:
# Reliability overlay + accuracy-vs-confidence bars across models.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]; ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="perfect")
for m in present:
    d = cal_all[cal_all.model == m]
    pts = reliability_curve(d.confidence.values, d.correct.values)
    if pts:
        xs, ys, _ = zip(*pts)
        ax.plot(xs, ys, "-o", color=MCOLOR[m], label=f"{m} (acc={d.correct.mean():.2f}, ECE={m_ece(d):.2f})")
ax.set_xlabel("confidence (1 - normalized entropy)"); ax.set_ylabel("accuracy")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.legend(fontsize=8); ax.set_title("Reliability by model")

ax = axes[1]
x = np.arange(len(present)); w = 0.38
ax.bar(x - w/2, [cal_all[cal_all.model == m].correct.mean() for m in present], w, label="accuracy")
ax.bar(x + w/2, [cal_all[cal_all.model == m].confidence.mean() for m in present], w, label="mean confidence")
ax.set_xticks(x); ax.set_xticklabels(present, rotation=20, ha="right", fontsize=8)
ax.set_ylim(0, 1); ax.legend(); ax.set_title("Accuracy vs mean confidence by model")
fig.tight_layout(); fig.savefig(OUT_DIR / "comparison_by_model.png", dpi=130, bbox_inches="tight"); plt.show()

### Rank-reversal check: this distribution vs the other

Loads the per-model scores saved by the *other* notebook
(`metrics_by_model_scored.csv`) and checks, for each metric, whether the model
ordering flips between the standard (MMLU-Pro) and target (REFLECT-failure)
distributions. A reversal that shows up under ECE **and** under the proper scores
(Brier/NLL) is robust; one that appears only under ECE is likely a binning artefact.
Run the scoring cell in *both* notebooks first.

In [ ]:
# === Rank-reversal check: this distribution vs the other =====================
from reflect.core.paths import analysis_experiment_dir
THIS_LABEL = "standard (MMLU-Pro)"
OTHER_DIR  = analysis_experiment_dir("llm_calibration_target_distribution")
other_csv  = OTHER_DIR / "metrics_by_model_scored.csv"

if not other_csv.exists():
    print(f"Run the scoring cell in the OTHER notebook first to produce:\n  {other_csv}")
else:
    other = pd.read_csv(other_csv, index_col=0)
    common = [m for m in score.index if m in other.index]
    print(f"this = {THIS_LABEL}")
    print(f"common models: {common}")
    if len(common) < 2:
        print("Need >= 2 models present in BOTH notebooks to detect a reversal. "
              "(Tip: run the gpt-5.4 retry cell so it has usable data in both.)")
    else:
        any_rev = False
        for metric in ["ece", "ece_w", "brier", "brier_w", "nll", "nll_w"]:
            a = score.loc[common, metric].sort_values()
            b = other.loc[common, metric].sort_values()
            ra = {m: i for i, m in enumerate(a.index)}
            rb = {m: i for i, m in enumerate(b.index)}
            rev = [(x, y) for i, x in enumerate(common) for y in common[i + 1:]
                   if (ra[x] - ra[y]) * (rb[x] - rb[y]) < 0]
            any_rev = any_rev or bool(rev)
            print(f"[{metric:8s}] this={list(a.index)}  other={list(b.index)}  "
                  f"-> {'RANK REVERSAL ' + str(rev) if rev else 'same order'}")
        print("\nVERDICT:", "reversal present" if any_rev else "no reversal in the current data",
              "| a reversal is only credible if it holds under brier/nll too, not ECE alone.")


## 11. Save results

In [ ]:
# One row per question (with options + ground truth + panel difficulty) for reproducibility.
q_records = []
for q in questions:
    q_records.append({"qid": q.qid, "category": q.category, "stem": q.stem,
                      "n_options": q.n_options, "options": q.options, "answer": q.answer,
                      "solvability": solvability[q.qid],
                      "difficulty": difficulty_bucket(solvability[q.qid])})
(OUT_DIR / "questions.jsonl").write_text("\n".join(json.dumps(r) for r in q_records))

df_all.to_csv(OUT_DIR / "results.csv", index=False)
by_cat.to_csv(OUT_DIR / "metrics_by_category.csv", index=False)
by_diff.to_csv(OUT_DIR / "metrics_by_difficulty.csv", index=False)

print("saved to", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print("  ", p.name)